##### Copyright 2025 Google LLC

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# 使用 ShieldGemma 2 和 Hugging Face Transformers 評估內容安全性

<table class="tfo-notebook-buttons" align="left"> <td>    <a target="_blank" href="https://ai.google.dev/responsible/docs/safeguards/shieldgemma"><img src="https://ai.google.dev/static/site-assets/images/docs/notebook-site-button.png" height="32" width="32" />View on ai.google.dev</a>
</td>    <td>
    <a target="_blank" href="https://colab.research.google.com/github/google-gemini/gemma-cookbook/blob/main/responsible/shieldgemma2_on_huggingface.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
</td> <td>    <a target="_blank" href="https://kaggle.com/kernels/welcome?src=https://github.com/google-gemini/gemma-cookbook/blob/main/responsible/shieldgemma2_on_huggingface.ipynb"><img src="https://www.kaggle.com/static/images/logos/kaggle-logo-transparent-300.png" height="32" width="70"/>Run in Kaggle</a>
</td> <td>    <a target="_blank" href="https://console.cloud.google.com/vertex-ai/colab/import/https%3A%2F%2Fraw.githubusercontent.com%2Fgoogle-gemini%2Fgemma-cookbook%2Fmain%2Fresponsible%2Fshieldgemma2_on_huggingface.ipynb"><img src="https://ai.google.dev/images/cloud-icon.svg" width="40" />Open in Vertex AI</a>
</td> <td>    <a target="_blank" href="https://github.com/google-gemini/gemma-cookbook/blob/main/responsible/shieldgemma2_on_huggingface.ipynb"><img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub</a>
</td>
</table>

**ShieldGemma 2** 模型經過訓練可以偵測[模型卡](https://ai.google.dev/gemma/docs/shieldgemma/model_card_2) 中詳細說明的關鍵危害。本指南示範如何使用 Hugging Face Transformers 建立可靠的資料和模型。
請注意，`ShieldGemma 2` 經過訓練，一次只能對一種傷害類型進行分類，因此您需要針對要檢查的每種傷害類型單獨調用 `ShieldGemma 2`。您可能也可以在 `ShieldGemma 2` 上使用模型調整技術。

# 支援的安全檢查

**ShieldGemma2** 是在 Gemma 3 的 4B IT checkpoint 上訓練的模型，經過訓練可以檢測和預測下列關鍵危害類型的違規行為：
* **危險內容**：圖像不得包含促進或鼓勵可能造成現實世界傷害的活動的內容（例如製造槍支和爆炸裝置、宣揚恐怖主義、自殺指導）。

* **露骨的性行為**：圖像不得包含描繪露骨或露骨性行為的內容（例如色情內容、色情裸體、強姦或性侵犯的描述）。

* **暴力/血腥**：圖片不得包含描繪令人震驚、聳人聽聞或無端暴力的內容（例如，過度血腥、針對動物的無端暴力、極度傷害或死亡時刻）。

這是基礎，但使用者可以提供客製化的安全策略作為模型的輸入，從而實現細粒度的控制和特定的用例要求。

# 支援的用例

ShieldGemma 2 應用作視覺語言模型的輸入濾鏡或影像產生系統的輸出濾鏡或兩者兼具。 ** ShieldGemma 2 具有以下主要優勢：
* **策略感知分類**：ShieldGemma 2 接受使用者定義的安全策略和影像作為輸入，為真實影像和產生的影像提供分類，並根據特定的策略指南進行客製化。
* **基於機率的輸出和閾值**：ShieldGemma 2 輸出其預測的機率分數，允許下游使用者根據其特定用例和風險承受能力靈活調整分類閾值。這使得安全分類的方法更加細緻、適應性更強。

輸入/輸出格式如下：* **輸入**：圖像+帶有策略定義的提示指令
* **輸出**：「是」/「否」tokens 的機率，「是」表示該影像違反了特定政策。 「是」token 的得分越高，模型對影像違反指定策略的置信度就越高。

# 使用範例

In [ ]:
# @title install Hugging Face Transformers v4.50+
! pip install -q 'transformers>=4.50.0'

In [ ]:
# @title Authenticate with Hugging Face Hub
# @markdown ShieldGemma is a gated model. To access the weights, you must accept
# @markdown the license on Hugging Face Hub under your account and then provide
# @markdown an [Access Token](https://huggingface.co/docs/hub/en/security-tokens)
# @markdown to authenticate with the Hugging Face Hub API. If using Colab, the
# @markdown easiest way to do this is by creating a read-only token specifically
# @markdown for Colab and setting this as the value of the `HF_TOKEN` secret;
# @markdown this token will then be reusable across all Colab notebooks. Other
# @markdown Python notebook platforms may provide a similar mechanism. For those
# @markdown that do not, un-comment the lines in this cell to install the
# @markdown Hugging Face Hub CLI and log in interactively.
# ! pip install -q 'huggingface_hub[cli]'
# ! huggingface-cli login

In [ ]:
from transformers import AutoProcessor, AutoModelForImageClassification
import torch

model_id = "google/shieldgemma-2-4b-it"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForImageClassification.from_pretrained(model_id)
model.to(torch.device("cuda"))

In [ ]:
from PIL import Image
import requests

# The image included in this Colab is benign and will not violate any of
# ShieldGemma's built-in content policies. Change this URL or otherwise update
# this code to use an image that may be violative.
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/bee.jpg"
image = Image.open(requests.get(url, stream=True).raw)

In [ ]:
inputs = processor(images=[image], return_tensors="pt").to(torch.device("cuda"))

with torch.no_grad():
  scores = model(**inputs)

# `scores` is a `ShieldGemma2ImageClassifierOutputWithNoAttention` instance
# continaing the logits and probabilities associated with the model predicting
# the `Yes` or `No` tokens as the response to the prompt batch, captured in the
# following properties.
#
#   *   `logits` (`torch.Tensor` of shape `(batch_size, 2)`): The first position
#       along dim=1 is the logits for the `Yes` token and the second position
#       along dim=1 is the logits for the `No` token.
#   *   `probabilities` (`torch.Tensor` of shape `(batch_size, 2)`): The first
#       position along dim=1 is the probability of predicting the `Yes` token
#       and the second position along dim=1 is the probability of predicting the
#       `No` token.
#
# When used with the `ShieldGemma2Processor`, the `batch_size` will be equal to
# `len(images) * len(policies)`, and the order within the batch will be
# img1_policy1, ... img1_policyN, ... imgM_policyN.
print(scores.logits)
print(scores.probabilities)

# ShieldGemma prompts are constructed such that predicting the `Yes` token means
# the content violates the policy. If you are only interested in the violative
# condition, you can extract only that slice from the output tensors.
p_violated = scores.probabilities[:, 0]
print(p_violated)
